# 社交媒体求救信息自动提取与定位 — 演示

本 notebook 端到端演示项目流程, 适合录制 3-6 分钟演示录像。

**流程**:
1. 数据构建与标注 (展示)
2. BERT NER 模型加载与单条推理
3. 地理编码
4. 求救点分布图可视化

## 0. 环境与路径

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.config import print_config
print_config()

## 1. 数据: 原始求救文本 + BIO 标注

In [ ]:
from src.data.annotate import load_jsonl
from src.config import TRAIN_FILE

train = load_jsonl(TRAIN_FILE)
print(f'训练集样本数: {len(train)}')
s = train[0]
print(f'\n原文: {s["text"]}')
print(f'实体: {[(e["type"], e["text"]) for e in s["entities"]]}')
print(f'BIO : {"".join(t[0] if t!="O" else "·" for t in s["tags"])}')

## 2. BERT NER 推理

输入一条求救文本, 自动抽取 4 类实体。

In [ ]:
from src.infer import infer_text, print_result

text = '求助! 汶川映秀镇中心学校3楼塌了,我们一家三口困在里面,缺水和食物,电话13900001234'
r = infer_text(text)
print_result(r)

## 3. 评估结果 (BERT vs 规则基线)

需要先运行 `python -m src.train` 和 `python -m src.evaluate`。

In [ ]:
import json
from src.config import METRICS_FILE

if METRICS_FILE.exists():
    m = json.loads(METRICS_FILE.read_text(encoding='utf-8'))
    print(f"BERT  F1: {m['bert']['f1']:.4f}  P: {m['bert']['precision']:.4f}  R: {m['bert']['recall']:.4f}")
    print(f"规则  F1: {m['rule_based']['f1']:.4f}  P: {m['rule_based']['precision']:.4f}  R: {m['rule_based']['recall']:.4f}")
    print(f'\n提升: +{(m["bert"]["f1"] - m["rule_based"]["f1"])*100:.1f} 个百分点')
else:
    print(f'未找到 {METRICS_FILE}, 请先运行训练与评估')

## 4. 求救点分布图

对全量数据批量推理, 生成交互式 HTML 地图。

In [ ]:
# 先批量推理导出 CSV (如未做过)
from src.config import STRUCTURED_CSV
if not STRUCTURED_CSV.exists():
    from src.infer import batch_infer
    batch_infer()

# 生成地图
from src.visualize import main as viz_main
viz_main()

In [ ]:
# 在 notebook 中直接展示地图
from src.config import RESCUE_MAP_FILE
import IPython
iframe = f'<iframe src="{RESCUE_MAP_FILE.as_posix()}" width="100%" height="500"></iframe>'
IPython.display.HTML(iframe)

## 5. 自定义文本推理
尝试输入你自己的求救文本。

In [ ]:
custom = '十万火急! 雅安芦山县发生6级地震,村里8户人家被困,急需饮用水和食物,联系13800005555'
r = infer_text(custom)
print_result(r)